# 00 CPU preflight — do this BEFORE enabling GPU

Accelerator: **None**. Internet: optional.
Attach Dataset `council-qwen05` (or official Qwen 0.5B Instruct with all 9 files).
Also attach this pack (`artifact_gate.py`, `smoke_scorer.py`, `datasets/smoke-v2.jsonl`).
Do not put tokens in cells.

In [ ]:
import json, shutil, sys
from pathlib import Path
WORKING = Path("/kaggle/working")
WORKING.mkdir(exist_ok=True)
# copy pack scripts next to us
for name in ("artifact_gate.py", "smoke_scorer.py", "worker_entry.py"):
    hits = list(Path("/kaggle/input").rglob(name)) + list(Path(".").glob(name))
    if not hits:
        raise FileNotFoundError(name + " — add the kaggle_pack input")
    shutil.copy2(hits[0], WORKING / name)
sys.path.insert(0, str(WORKING))
import artifact_gate, smoke_scorer

required = {
    "model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json",
    "generation_config.json", "vocab.json", "merges.txt", "LICENSE",
}
cands = []
for cfg in Path("/kaggle/input").rglob("config.json"):
    root = cfg.parent
    if all((root / n).is_file() for n in required):
        cands.append(root)
print("complete candidates:", [str(c) for c in cands])
if len(cands) != 1:
    raise RuntimeError(f"expected exactly one complete model dir, found {len(cands)}")
MODEL_DIR = cands[0]
print("MODEL_DIR", MODEL_DIR)
(WORKING / "model_dir.txt").write_text(str(MODEL_DIR))

import subprocess
r = subprocess.run([sys.executable, str(WORKING / "artifact_gate.py"), str(MODEL_DIR)])
print("artifact_gate rc", r.returncode)
if r.returncode != 0:
    raise SystemExit("file-gate failed — stay on CPU")

from transformers import AutoConfig, AutoTokenizer
cfg = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=False)
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=False)
print("model_type", cfg.model_type, "tokenizer", type(tok).__name__, "vocab", len(tok))

bench = next(Path("/kaggle/input").rglob("smoke-v2.jsonl"), Path("datasets/smoke-v2.jsonl"))
items = smoke_scorer.load_smoke(str(bench))
smoke_scorer.validate_benchmark(items)
print("smoke-v2 records", len(items))
print("CPU PREFLIGHT PASS")